In [3]:


"""
Portfolio Project: Web Scraping + Data Cleaning + Visualization
Source: books.toscrape.com (a public practice site, scraping allowed)
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

BASE_URL = "http://books.toscrape.com/catalogue/page-{}.html"

books = []

# Scrape first 5 pages (~100 books)
for page in range(1, 6):
    url = BASE_URL.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    book_items = soup.find_all("article", class_="product_pod")

    for item in book_items:
        title = item.h3.a["title"]
        price_text = item.find("p", class_="price_color").text
        price = float(re.sub(r"[^\d.]", "", price_text))

        rating_class = item.find("p", class_="star-rating")["class"][1]
        rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
        rating = rating_map.get(rating_class, 0)

        availability = item.find("p", class_="instock availability").text.strip()
        in_stock = "In stock" in availability

        books.append({
            "Title": title,
            "Price (GBP)": price,
            "Rating": rating,
            "In Stock": in_stock
        })

    print(f"Page {page} done. Total books so far: {len(books)}")

print(f"\nScraped {len(books)} books successfully.")

# ---- Convert to DataFrame ----
df = pd.DataFrame(books)

print("\nRaw scraped data sample:")
print(df.head())

# ---- Data Cleaning ----
df.drop_duplicates(subset="Title", inplace=True)
df["Title"] = df["Title"].str.strip()
df.dropna(inplace=True)

print(f"\nAfter cleaning: {len(df)} unique books remain.")

# Save cleaned data
df.to_csv("books_cleaned.csv", index=False)
print("\nSaved cleaned data to books_cleaned.csv")
print(df.describe())

Page 1 done. Total books so far: 20
Page 2 done. Total books so far: 40
Page 3 done. Total books so far: 60
Page 4 done. Total books so far: 80
Page 5 done. Total books so far: 100

Scraped 100 books successfully.

Raw scraped data sample:
                                   Title  Price (GBP)  Rating  In Stock
0                   A Light in the Attic        51.77       3      True
1                     Tipping the Velvet        53.74       1      True
2                             Soumission        50.10       1      True
3                          Sharp Objects        47.82       4      True
4  Sapiens: A Brief History of Humankind        54.23       5      True

After cleaning: 100 unique books remain.

Saved cleaned data to books_cleaned.csv
       Price (GBP)      Rating
count   100.000000  100.000000
mean     34.560700    2.930000
std      14.638531    1.423149
min      10.160000    1.000000
25%      19.897500    2.000000
50%      34.775000    3.000000
75%      47.967500    4.0000